# Part 3 — Model Baseline: LightGBM & XGBoost

**Train:** 2023–2024 (~16.4M rows)  
**Test:** 2025 (~4.2M rows)  
**Split strategy:** Chronological (no shuffled CV across years — prevents leakage)  

We train LightGBM as the primary new model (better histogram binning at 20M rows) and XGBoost as a cross-part baseline.

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import config

config.assert_data_exists()

## 1. Load Processed Features

In [ ]:
feat_path = config.DATA_PART3_PROCESSED / "flights_2023_2025_features.parquet"
df = pd.read_parquet(feat_path)
print(df.shape)
df.head()

## 2. Train / Test Split (Chronological)

In [ ]:
TARGET = "ARR_DEL15"
CAT_COLS = ["ORIGIN", "DEST", "OP_CARRIER"]

FEATURES = [c for c in df.columns if c not in [TARGET, "year", "YEAR"]]

train = df[df["YEAR"] <= 2024]
test  = df[df["YEAR"] == 2025]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train delay rate: {y_train.mean():.3f}  |  Test delay rate: {y_test.mean():.3f}")

## 3. Encode Categoricals

In [ ]:
# LightGBM handles categoricals natively; encode to 'category' dtype
for col in CAT_COLS:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_test[col]  = X_test[col].astype("category")

## 4. LightGBM Baseline

In [ ]:
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "n_jobs": -1,
    "verbose": -1,
    "random_state": 42,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

In [ ]:
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
lgb_pred = (lgb_prob >= 0.5).astype(int)

print("=== LightGBM ===")
print(f"Accuracy: {accuracy_score(y_test, lgb_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, lgb_prob):.4f}")
print(f"F1:       {f1_score(y_test, lgb_pred):.4f}")
print(classification_report(y_test, lgb_pred))

## 5. XGBoost Baseline

In [ ]:
# XGBoost doesn't handle category dtype — encode to integer codes
X_train_xgb = X_train.copy()
X_test_xgb  = X_test.copy()
for col in CAT_COLS:
    if col in X_train_xgb.columns:
        X_train_xgb[col] = X_train_xgb[col].cat.codes
        X_test_xgb[col]  = X_test_xgb[col].cat.codes

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    early_stopping_rounds=30,
    tree_method="hist",  # fast for large datasets
    device="cuda",        # remove if no GPU
    n_jobs=-1,
    random_state=42,
    verbosity=1,
)
xgb_model.fit(X_train_xgb, y_train, eval_set=[(X_test_xgb, y_test)], verbose=100)

In [ ]:
xgb_prob = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_pred = (xgb_prob >= 0.5).astype(int)

print("=== XGBoost ===")
print(f"Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, xgb_prob):.4f}")
print(f"F1:       {f1_score(y_test, xgb_pred):.4f}")

## 6. Feature Importance (LightGBM)

In [ ]:
feat_imp = pd.Series(lgb_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
feat_imp.head(20).plot(kind="barh", figsize=(8, 8), title="LightGBM Feature Importance (top 20)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Save Models

In [ ]:
import joblib

out_dir = config.DATA_PART3_PROCESSED
lgb_model.booster_.save_model(str(out_dir / "lgb_baseline.txt"))
joblib.dump(xgb_model, out_dir / "xgb_baseline.joblib")
print("Models saved.")